# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library and the Croissant metadata standard. You will learn how to access the schema, review its structure, extract records, perform basic analysis, and visualize data—all with robust entity referencing through Croissant `@id` fields.

### Dataset Source

Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Let's load the Croissant schema and the dataset package metadata with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object (not a dictionary)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview

Let's review the available **record sets** (`cr:RecordSet`), their fields, and their Croissant `@id`s for entity-based referencing.

First, we'll list record sets with their `@id` and a short description, then show which fields/columns they contain (also by Croissant `@id`).

In [ ]:
# List available record sets and their @id's
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name','(no name)')}")
    print(f"  description: {rs.get('description','(no description)')}")
    fields = rs.get('field', [])
    # Ensure fields is always a list
    if isinstance(fields, dict): fields = [fields]
    print("  Fields/Columns (@id):")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id')}")
        else:
            print(f"    - {field}")
    print()

> **Tip:** Use the `@id` fields found above when extracting data for maximum robustness and traceability.

## 3. Data Extraction

Next, we'll load one or more **record sets** (using their `@id`) into pandas DataFrames. We'll demonstrate how to extract records for a chosen table and display its columns (by `@id`).

In [ ]:
# List all record set @id's for reference
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set @id's:")
for rid in record_set_ids:
    print(f" - {rid}")

# For this dataset, assume there is one main record set. Update if there are more.
# Set the record_set_id variable by choosing from the printed IDs.
record_set_id = record_set_ids[0]  # Select the main/open data table

# Extract records as list of dicts from this record set
data = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(data)

print(f"Columns for record set {record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Now we will:
- Select a **numeric field** (using its `@id` as column name), filter by value,
- Normalize the value,
- Group by a **categorical field** (also by `@id`), and show aggregate statistics.

**Note:** You can see the column `@id`s in the output above.

In [ ]:
# --- CHANGE THESE to match available @id fields in your DataFrame (see above) ---
# Example: Assume '@id' for Age is 'http://mlcommons.org/croissant/Field/age' and Sex is 'http://mlcommons.org/croissant/Field/sex'
numeric_field_id = [col for col in df.columns if 'age' in col.lower() or 'Age' in col][0]  # Try to auto-select Age field
group_field_id = [col for col in df.columns if 'sex' in col.lower() or 'Sex' in col or 'gender' in col.lower()][0]  # Try to auto-select Sex or Gender

print(f"Numeric field: {numeric_field_id}")
print(f"Group/categorical field: {group_field_id}\n")

# Convert numeric field to float if not already
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records with age > threshold
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id and calculate mean statistics
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df)
else:
    print(f"Group field {group_field_id} not present in columns.")

## 5. Visualization

We'll plot the distribution of the numeric field (e.g., Age), colored or separated by the group field (e.g., Sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8,5))
if group_field_id in filtered_df.columns:
    sns.histplot(data=filtered_df, x=numeric_field_id, hue=group_field_id, kde=True, bins=15, palette='Set1', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
else:
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the [Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://doi.org/10.71728/senscience.qs2f-h81p) dataset following the Croissant metadata standard, using the `mlcroissant` library;
- Explored record sets, fields, and referenced all data by their Croissant `@id`;
- Extracted tabular records and performed filtering, normalization, grouping, and visualization;
- Gained insights into the dataset structure and example clinical variables (such as age distribution by sex).

For further analysis:
- Investigate other fields (e.g., comorbidities, treatment types, MSI-H status),
- Apply advanced modeling and statistics,
- Reference entities by their `@id` to ensure reproducibility and metadata traceability.

_For more, visit the [mlcroissant documentation](https://mlcommons.github.io/croissant)._